<a href="https://colab.research.google.com/github/crishanser-beep/desktop-tutorial/blob/main/Mini_Projeto_Cristina_Aigner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import data_table
data_table.enable_dataframe_formatter()



import pandas as pd


df = pd.read_csv('Base Varejo.csv', sep=';')
df.head ()

,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,01/02/2019,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA,NaN,NaN,NaN,NaN
1,01/02/2019,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS,NaN,NaN,NaN,NaN
2,01/02/2019,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO,NaN,NaN,NaN,NaN
3,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,NaN,NaN,NaN,NaN
4,01/02/2019,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO,NaN,NaN,NaN,NaN


In [ ]:
columns_to_drop = ['Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']
df = df.drop(columns=columns_to_drop, errors='ignore')
df.head()

,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME
0,01/02/2019,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA
1,01/02/2019,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS
2,01/02/2019,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO
3,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI
4,01/02/2019,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO


In [ ]:
df['PR_CAT'].value_counts()

,count
PR_CAT,
ALIMENTOS,434767
HIGIENE,155574
LIMPEZA,145754
BEBIDAS,43299
PET,32399
ACESSORIOS,14557
#N/D,3650


In [ ]:
percentage_distribution = (df['PR_CAT'].value_counts() / len(df)) * 100
display(percentage_distribution)

,count
PR_CAT,
ALIMENTOS,52.381566
HIGIENE,18.743855
LIMPEZA,17.560723
BEBIDAS,5.216747
PET,3.903494
ACESSORIOS,1.753855
#N/D,0.439759


In [ ]:
df = df[df['PR_CAT'] != '#N/D']
df['PR_CAT'].value_counts()

,count
PR_CAT,
ALIMENTOS,434767
HIGIENE,155574
LIMPEZA,145754
BEBIDAS,43299
PET,32399
ACESSORIOS,14557


In [ ]:
print('--- Verificação de Valores Nulos por Coluna ---\n')
null_values = df.isnull().sum()
print(null_values[null_values > 0])

print('\n--- Verificação de Linhas Duplicadas ---\n')
duplicate_rows = df.duplicated().sum()
print(f'Número de linhas duplicadas: {duplicate_rows}')

print('\n--- Verificação de Inconsistências (Datas e Categorias) ---\n')
# Verificação de datas inválidas na coluna 'DATA'
# Converte a coluna 'DATA' para datetime, tratando erros para identificar datas inválidas
df['DATA_CONVERTIDA'] = pd.to_datetime(df['DATA'], errors='coerce', format='%d/%m/%Y')
invalid_dates = df[df['DATA_CONVERTIDA'].isnull()]['DATA'].unique()
if len(invalid_dates) > 0:
    print(f'Foram encontradas {len(invalid_dates)} datas inválidas na coluna "DATA". Exemplos: {invalid_dates[:5]}')
else:
    print('Nenhuma data inválida encontrada na coluna "DATA".')

# Verificação de categorias vazias ou inconsistentes (além de #N/D, que já foi tratado)
# Vamos verificar se existem strings vazias ou espaços em branco em 'PR_CAT'
inconsistent_pr_cat = df[df['PR_CAT'].isin(['', ' '])]['PR_CAT'].count()
if inconsistent_pr_cat > 0:
    print(f'Foram encontradas {inconsistent_pr_cat} entradas vazias ou com apenas espaços na coluna "PR_CAT".')
else:
    print('Nenhuma categoria vazia ou com apenas espaços encontrada na coluna "PR_CAT" (além do #N/D, que já foi removido).')

# Remove a coluna temporária criada para a verificação de datas
df = df.drop(columns=['DATA_CONVERTIDA'], errors='ignore')

--- Verificação de Valores Nulos por Coluna ---

Series([], dtype: int64)

--- Verificação de Linhas Duplicadas ---

Número de linhas duplicadas: 96131

--- Verificação de Inconsistências (Datas e Categorias) ---

Nenhuma data inválida encontrada na coluna "DATA".
Nenhuma categoria vazia ou com apenas espaços encontrada na coluna "PR_CAT" (além do #N/D, que já foi removido).


In [ ]:
df_cleaned = df.drop_duplicates()
print(f'Número de linhas após remover duplicatas: {df_cleaned.shape[0]}')

Número de linhas após remover duplicatas: 730219


### Análise de Padrões de Agrupamento

In [ ]:
print('--- Vendas por Gênero do Cliente (CL_GENERO) ---\n')
sales_by_gender = df.groupby('CL_GENERO')['CO_ID'].count().reset_index()
sales_by_gender.rename(columns={'CO_ID': 'Numero_de_Vendas'}, inplace=True)
display(sales_by_gender.sort_values(by='Numero_de_Vendas', ascending=False))

--- Vendas por Gênero do Cliente (CL_GENERO) ---



,CL_GENERO,Numero_de_Vendas
0,F,380735
1,M,349484


In [ ]:
print('\n--- Vendas por Categoria de Produto (PR_CAT) ---\n')
sales_by_category = df.groupby('PR_CAT')['CO_ID'].count().reset_index()
sales_by_category.rename(columns={'CO_ID': 'Numero_de_Vendas'}, inplace=True)
display(sales_by_category.sort_values(by='Numero_de_Vendas', ascending=False))


--- Vendas por Categoria de Produto (PR_CAT) ---



,PR_CAT,Numero_de_Vendas
1,ALIMENTOS,384197
3,HIGIENE,137702
4,LIMPEZA,128632
2,BEBIDAS,38264
5,PET,28553
0,ACESSORIOS,12871


Agora, vamos analisar um agrupamento mais detalhado: vendas por gênero e categoria de produto.

In [ ]:
print('\n--- Vendas por Gênero e Categoria de Produto ---\n')
sales_by_gender_category = df.groupby(['CL_GENERO', 'PR_CAT'])['CO_ID'].count().reset_index()
sales_by_gender_category.rename(columns={'CO_ID': 'Numero_de_Vendas'}, inplace=True)
display(sales_by_gender_category.sort_values(by=['CL_GENERO', 'Numero_de_Vendas'], ascending=[True, False]))


--- Vendas por Gênero e Categoria de Produto ---



,CL_GENERO,PR_CAT,Numero_de_Vendas
1,F,ALIMENTOS,200274
3,F,HIGIENE,71721
4,F,LIMPEZA,67328
2,F,BEBIDAS,19764
5,F,PET,14809
0,F,ACESSORIOS,6839
7,M,ALIMENTOS,183923
9,M,HIGIENE,65981
10,M,LIMPEZA,61304
8,M,BEBIDAS,18500


In [ ]:
# Atualiza o DataFrame principal 'df' com os dados sem duplicatas
df = df_cleaned.copy()

# Converte a coluna 'DATA' para o tipo datetime
# Já foi validado anteriormente que não há datas inválidas graves, então 'coerce' não deve causar perdas
df['DATA'] = pd.to_datetime(df['DATA'], format='%d/%m/%Y', errors='coerce')

# Verifica o tipo de dados da coluna 'DATA' após a conversão
print('\n--- Verificação do Tipo de Dados da Coluna DATA ---\n')
print(df['DATA'].dtype)


--- Verificação do Tipo de Dados da Coluna DATA ---

datetime64[ns]


In [ ]:
print('--- Estatísticas Descritivas para a Coluna CL_FHL (Número de Filhos) ---\n')

cl_fhl_mean = df['CL_FHL'].mean()
cl_fhl_median = df['CL_FHL'].median()
cl_fhl_std = df['CL_FHL'].std()
cl_fhl_mode = df['CL_FHL'].mode()[0] # mode() pode retornar múltiplos valores, pegamos o primeiro
cl_fhl_max = df['CL_FHL'].max()
cl_fhl_min = df['CL_FHL'].min()
cl_fhl_count = df['CL_FHL'].count()

print(f'Média (CL_FHL): {cl_fhl_mean:.2f}')
print(f'Mediana (CL_FHL): {cl_fhl_median:.2f}')
print(f'Desvio Padrão (CL_FHL): {cl_fhl_std:.2f}')
print(f'Moda (CL_FHL): {cl_fhl_mode}')
print(f'Máximo (CL_FHL): {cl_fhl_max}')
print(f'Mínimo (CL_FHL): {cl_fhl_min}')
print(f'Contagem (CL_FHL): {cl_fhl_count}')

--- Estatísticas Descritivas para a Coluna CL_FHL (Número de Filhos) ---

Média (CL_FHL): 1.15
Mediana (CL_FHL): 0.00
Desvio Padrão (CL_FHL): 1.42
Moda (CL_FHL): 0
Máximo (CL_FHL): 4
Mínimo (CL_FHL): 0
Contagem (CL_FHL): 730219
